# Deep Agents — Backends 워크스루

**2주차 (jh-lee)** — 같은 에이전트 코드, 다른 저장 매체. `backend=` 인자 한 줄로 데모에서 운영까지 옮긴다.

## 이 노트북에서 보여주는 것

deepagents 의 4개 빌트인 백엔드를 한 자리에서 비교하고, 그 위에 거는 권한 제어(policy hooks) 까지 한 데모로 본다.

1. **Demo 1 — StateBackend** (휘발성, thread-scoped) — 가장 단순한 디폴트
2. **Demo 2 — FilesystemBackend** (호스트 디스크 + `virtual_mode`) — sandbox 모드
3. **Demo 3 — StoreBackend** (LangGraph Store 영속, thread-cross) — 장기 메모리
4. **Demo 4 — CompositeBackend** (라우팅) — 위 셋을 한 에이전트에서 동시에
5. **Demo 5 — Policy hooks** (GuardedBackend + PolicyWrapper) — 백엔드 위에 권한 제어

각 Demo 셀 위의 markdown 이 *시연 흐름*, *관전 포인트*, *왜 그런가* 를 설명한다. 자세한 원리·코드 인용은 같은 디렉토리의 교안(`archives/source/01_textbook.md`)·슬라이드(`archives/source/slides.md`) 참조.

## 사전 준비

```bash
cd /Users/jaden/projects/langchain-docs/deep-agents/week2-backend-jh-lee
cp .env_sample .env   # 필요하면 키 채워넣기
pip install -r scripts/requirements.txt

# Ollama 데몬 실행 + 모델 다운로드 (한 번만)
ollama pull gemma4:e2b
```

In [7]:
import sys
print(sys.executable)

/Users/jaden/.pyenv/versions/3.12.10/bin/python


In [8]:
import sys
!{sys.executable} -m pip install langchain-ollama python-dotenv deepagents langchain langgraph langgraph-checkpoint

  Using cached langchain_ollama-1.1.0-py3-none-any.whl.metadata (3.0 kB)
  Using cached deepagents-0.6.1-py3-none-any.whl.metadata (4.3 kB)
  Using cached langchain-1.3.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached langgraph-1.2.0-py3-none-any.whl.metadata (8.0 kB)
  Using cached langgraph_checkpoint-4.1.0-py3-none-any.whl.metadata (5.2 kB)
  Using cached langchain_core-1.4.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached ollama-0.6.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached langsmith-0.8.4-py3-none-any.whl.metadata (15 kB)
  Using cached langchain_anthropic-1.4.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached langchain_google_genai-4.2.2-py3-none-any.whl.metadata (2.7 kB)
  Using cached wcmatch-10.1-py3-none-any.whl.metadata (5.1 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached langgraph_prebuilt-1.1.0-py3-none-any.whl.metadata (5.2 kB)
  Using cached langgraph_sdk-0.3.14-py3-none-any.whl.metadata (1.7 kB)
  Using cached xxhash-

## 공통 셋업 — Ollama 모델 초기화

다음 셀이 4개 Demo 모두에 공유될 LLM(`model` 변수) 을 만든다.

- **Ollama** 를 쓰는 이유: 로컬에서 키 없이 동작 + tool-use 지원. `deepagents` 는 LangChain 호환 model 이면 무엇이든 받으므로, 운영에서는 OpenAI·Anthropic 모델로 한 줄 교체 가능.
- **`OLLAMA_MODEL`** (환경변수): 모델 이름 오버라이드. 기본 `gemma4:e2b`.
- **`OLLAMA_BASE_URL`** (환경변수): 원격 Ollama 서버 주소 (선택). 미설정 시 로컬 `127.0.0.1:11434`.

**점검 포인트** (셀 실행 시 에러가 나면)
- `curl -s http://localhost:11434/api/tags` 응답 없으면 → Ollama 데몬이 꺼져 있음
- `ollama list | grep gemma` 가 비어 있으면 → `ollama pull gemma4:e2b` 필요
- 일부 모델 변종은 tool-use 미지원 — 교안 부록 C 참조

In [32]:
# 공통 셋업 — Ollama gemma4:e2b (사전에 `ollama pull gemma4:e2b`)
# 더 가벼운 모델로 기능 검증용: qwen3:0.6b (사전에 `ollama pull qwen3:0.6b`)

import os
from dotenv import find_dotenv, load_dotenv
load_dotenv(find_dotenv())

from langchain_ollama import ChatOllama
# OLLAMA_MODEL = os.environ.get('OLLAMA_MODEL', 'gemma4:31b')
# OLLAMA_MODEL = os.environ.get('OLLAMA_MODEL', 'gemma4:26b')
os.environ["OLLAMA_MODEL"] = "gemma4:e2b"
# os.environ["OLLAMA_MODEL"] = "qwen3:0.6b"

OLLAMA_MODEL = os.environ.get('OLLAMA_MODEL', 'gemma4:e2b')
extra = {'base_url': os.environ['OLLAMA_BASE_URL']} if os.environ.get('OLLAMA_BASE_URL') else {}
model = ChatOllama(model=OLLAMA_MODEL, **extra)

---

## 4종 백엔드 한 장 비교

> **도구는 6개로 고정** (`ls` / `read_file` / `write_file` / `edit_file` / `glob` / `grep`) — 그 아래 저장 매체만 `backend=` 가 결정한다.

|            | State    | Filesystem    | Store         | Composite |
|---         |---       |---            |---            |---        |
| **생애**   | 휘발     | 영속(디스크)  | 영속(store)   | 합성      |
| **Scope**  | thread   | 디스크        | 멀티 thread   | 라우팅    |
| **케이스** | scratch  | sandbox       | 장기 메모리   | 혼합      |

같은 에이전트 코드를 ephemeral 데모에서 멀티 테넌트 운영까지 한 인자 교체로 옮길 수 있다는 것이 본 추상화의 핵심이다. 아래 4개 Demo 가 표의 네 칸을 차례로 보여준다.

## Demo 1 — StateBackend (thread-scoped 휘발성)

> 같은 thread 안에서만 파일이 살아남는, 가장 단순한 디폴트 백엔드.

**무엇을 시연하나**
- Turn 1: `demo-thread-1` 에서 `/notes/note.md` 작성
- Turn 2: 같은 thread 에서 read → 노트 있음
- Turn 3: `demo-thread-2` 로 변경 후 read → 노트 없음 (휘발 증명)

**관전 포인트**
- Turn 2 출력에 "2주차 발표일은 2026-05-22" 가 보여야 한다
- Turn 3 출력에는 "파일이 없다" 류 메시지가 나와야 한다

**왜 그런가**
- `StateBackend` 는 데이터를 LangGraph state 채널(`files: {path: content}`) 에 둠
- thread 가 바뀌면 state 도 새로 시작 → 파일도 사라짐
- checkpointer 가 영속(PostgresSaver 등) 이면 프로세스가 죽어도 thread 가 살아남아 다음 실행에서 이어 씀 — 본 데모의 InMemorySaver 는 스크립트 종료 시 함께 휘발

→ 자세한 원리·코드 인용은 교안 §3 (`archives/source/01_textbook.md`)
→ 소스: `archives/scripts_py/01_state_backend.py`

In [34]:
!python -u ../archives/scripts_py/01_state_backend.py

▶ StateBackend 데모 — LangGraph state 채널에 파일 저장.
  · 같은 thread 안에서만 파일이 보이며, thread 가 끝나면 휘발.
  · checkpointer 가 영속이면 (예: PostgresSaver) 프로세스가 죽어도 thread 가 살아남아 다음 실행에서 이어 쓸 수 있음.
  · 본 데모는 InMemorySaver 라 스크립트 종료 시 thread state 도 함께 사라짐.

=== Turn 1: 기록 ===
  ↳ thread_id=demo-thread-1 으로 invoke → 파일은 이 thread 의 state 채널에 적재됨.
/Users/jaden/.pyenv/versions/3.12.10/lib/python3.12/site-packages/deepagents/middleware/filesystem.py:1700: LangChainDeprecationWarning: Passing a callable (factory) as `backend` is deprecated and will be removed in deepagents==0.7.0. Pass a `BackendProtocol` instance directly instead (e.g. `StateBackend()`).
  backend = self._get_backend(request.runtime)  # ty: ignore[invalid-argument-type]
/Users/jaden/projects/langchain-docs/deep-agents/week2-backend-jh-lee/scripts/../archives/scripts_py/01_state_backend.py:50: LangChainDeprecationWarning: Passing `runtime` to `StateBackend` is deprecated and will be removed in deepagents==0.7.0. `StateBackend` now reads and wri

## Demo 2 — FilesystemBackend (root_dir + virtual_mode)

> 호스트 디스크의 한 디렉토리를 sandbox 로. `virtual_mode` 한 플래그가 표면을 바꾼다.

**무엇을 시연하나**
- 임시 디렉토리를 root 로 잡고, 두 모드를 비교
- `virtual_mode=True`: 에이전트가 보는 `/report.md` → 호스트 `<root>/report.md` 로 정규화
- `virtual_mode=False`: 호스트 절대경로를 그대로 받아 그 자리에 저장
- 마지막에 `ls /` 로 두 파일을 한 자리에서 확인

**관전 포인트**
- 출력에 `sandbox root: /private/var/folders/.../tmp...` 가 찍힌다
- 두 모드 모두 호스트 디스크의 `<root>/{report,raw}.md` 가 `exists=True` 로 나와야 한다
- `ls /` 결과에 `/raw.md`, `/report.md` 두 줄이 모두 보여야 한다

**왜 그런가**
- `virtual_mode=True` 는 에이전트 입장에서 `/` 가 `root_dir` 인 가짜 파일시스템을 만들어 줌 → 보안 격리·symlink 차단 자동
- `virtual_mode=False` 는 호스트 경로를 그대로 노출 → 격리는 약하지만 시연 가독성이 좋음
- 운영 기본값은 `virtual_mode=True` (CI sandbox·컨테이너 격리에 적합)

→ 두 모드 비교표(표 3)는 교안 §4 참조
→ 소스: `archives/scripts_py/02_filesystem_backend.py`

In [35]:
!python -u ../archives/scripts_py/02_filesystem_backend.py

▶ FilesystemBackend 데모 — 호스트 디스크의 root_dir 아래에 실제 파일 생성.
  · 본 데모는 tempfile.TemporaryDirectory() 를 root 로 쓰므로 스크립트 종료 시 자동 삭제.
  · 운영에서 영속 디렉토리를 root 로 쓰면 파일이 살아남음.

sandbox root: /private/var/folders/rt/yfdr6vp909qbw4ybcq5bzffh0000gn/T/tmpld3k3fas

=== virtual_mode=True (경로 정규화) ===
  ↳ 에이전트는 '/report.md' 로 절대경로를 보지만, 실제로는 <root>/report.md 로 정규화되어 sandbox 안에 떨어짐.
호스트 디스크: /private/var/folders/rt/yfdr6vp909qbw4ybcq5bzffh0000gn/T/tmpld3k3fas/report.md exists=True content=# hello virtual

=== virtual_mode=False (호스트 절대경로 그대로) ===
  ↳ 에이전트가 호스트 절대경로를 그대로 받아 그 자리에 파일 생성 — 격리는 약하지만 직관적.
호스트 디스크: /private/var/folders/rt/yfdr6vp909qbw4ybcq5bzffh0000gn/T/tmpld3k3fas/raw.md exists=False content=N/A

=== ls / (virtual_mode=True 측에서) ===
  ↳ ls 결과의 경로(/raw.md, /report.md) 는 sandbox 시점. 실제 위치는 위 'sandbox root:' 아래.
ls(/)


## Demo 3 — StoreBackend (LangGraph Store 영속)

> thread 를 가로질러 살아남는 영속 저장소. LangGraph `BaseStore` 위에서 동작한다.

**무엇을 시연하나**
- session-A (`thread_id="session-A"`) 에서 `/memories/release.md` 에 "2주차 PDF 빌드는 build.py 로 한다" 작성
- session-B (다른 thread) 에서 같은 파일 read → 같은 내용이 그대로 읽힘
- 두 호출이 같은 `InMemoryStore()` 인스턴스를 공유하기 때문

**관전 포인트**
- session-A 단계에 작성 확인 메시지 ("기억했다" 류) 출력
- session-B 단계에 동일 내용 ("2주차 PDF 빌드는 `build.py` 로 수행") 회상 출력
- thread 가 바뀌었음에도 파일이 살아 있다는 점이 본 데모의 핵심

**왜 그런가**
- LangGraph 의 일반 지침: *checkpointer 는 thread 내, store 는 thread 간*
- `StoreBackend` 는 BaseStore 의 namespace + key + value 3축 영속 저장소를 빌려 씀
- 운영에서는 `InMemoryStore` → `PostgresStore` 한 줄 교체로 진짜 영속

→ namespace·의미 기반 검색은 교안 §5 참조
→ 소스: `archives/scripts_py/03_store_backend.py`

In [36]:
!python -u ../archives/scripts_py/03_store_backend.py

▶ StoreBackend 데모 — LangGraph BaseStore 위에 파일 저장. thread 를 가로질러 살아남음.
  · 본 데모는 InMemoryStore 라 스크립트 종료 시 휘발.
  · PostgresStore 로 바꾸면 진짜 영속.

=== session-A: 기록 ===
  ↳ thread_id=session-A 로 /memories/release.md 작성 → 파일은 store 의 namespace 키에 들어감.
/Users/jaden/.pyenv/versions/3.12.10/lib/python3.12/site-packages/deepagents/middleware/filesystem.py:1700: LangChainDeprecationWarning: Passing a callable (factory) as `backend` is deprecated and will be removed in deepagents==0.7.0. Pass a `BackendProtocol` instance directly instead (e.g. `StateBackend()`).
  backend = self._get_backend(request.runtime)  # ty: ignore[invalid-argument-type]
/Users/jaden/projects/langchain-docs/deep-agents/week2-backend-jh-lee/scripts/../archives/scripts_py/03_store_backend.py:52: LangChainDeprecationWarning: Passing `runtime` to `StoreBackend` is deprecated and will be removed in deepagents==0.7.0. `StoreBackend` now obtains store and context via `get_store()` / `get_runtime()`. Use `StoreBackend()` or `StoreB

## Demo 4 — CompositeBackend (라우팅 규칙)

> 한 에이전트, 세 매체 — 경로 prefix 가 어디로 갈지 결정한다.

**무엇을 시연하나**
- `/memories/note.md` → StoreBackend (thread-cross 영속)
- `/shared/draft.md` → FilesystemBackend (호스트 디스크, sandboxed)
- `/tmp/scratch.md` → default = StateBackend (휘발)
- 마지막에 호스트 디스크를 직접 들여다보며 `/shared/*` 만 떨어졌음을 확인

**관전 포인트**
- 호스트 디스크 검증 단계에서 `/shared/draft.md` 만 실제 파일로 존재해야 한다
- `/memories/*` 와 `/tmp/*` 는 호스트에 없어야 한다 (각각 store 와 state 에 있음)
- 에이전트가 `ls /` 를 부르면 세 백엔드 결과가 합쳐져 `/memories`, `/shared`, `/tmp` 가 한 자리에 보인다

**왜 그런가**
- `CompositeBackend(default=..., routes={...})` 가 prefix 기반으로 라우팅 — **longer prefix wins**
- `ls`/`glob`/`grep` 같은 디렉토리 횡단 호출은 모든 백엔드를 호출해 결과를 합치되, 원래 prefix 를 보존
- "백엔드 선택" 의 문제를 "경로 설계" 의 문제로 옮김 — 한 에이전트가 휘발·영속·디스크를 동시에 봄

→ 라우팅 알고리즘·결과 합치기는 교안 §6 참조
→ 소스: `archives/scripts_py/04_composite_backend.py`

In [37]:
!python -u ../archives/scripts_py/04_composite_backend.py

▶ CompositeBackend 데모 — 경로 prefix 기반 라우팅.
  · 같은 에이전트 안에서 세 매체(Store / Filesystem / State) 가 공존.
  · 매체별로 파일 수명이 다름 — 아래 step 별 설명 참조.

=== 1) /memories/note.md (Store 경로) write ===
  ↳ /memories/* → StoreBackend.
     저장 위치: LangGraph Store (본 데모는 InMemoryStore).
     수명: thread-cross 영속 — 다른 thread 에서도 보임. 프로세스 종료 시 휘발 (PostgresStore 로 바꾸면 영속).
/Users/jaden/.pyenv/versions/3.12.10/lib/python3.12/site-packages/deepagents/middleware/filesystem.py:1700: LangChainDeprecationWarning: Passing a callable (factory) as `backend` is deprecated and will be removed in deepagents==0.7.0. Pass a `BackendProtocol` instance directly instead (e.g. `StateBackend()`).
  backend = self._get_backend(request.runtime)  # ty: ignore[invalid-argument-type]
/Users/jaden/projects/langchain-docs/deep-agents/week2-backend-jh-lee/scripts/../archives/scripts_py/04_composite_backend.py:72: LangChainDeprecationWarning: Passing `runtime` to `StateBackend` is deprecated and will be removed in deepagents==0.7.0. `State

## Demo 5 — 권한 제어 (GuardedBackend + PolicyWrapper)

> 백엔드 위에 정책(deny) 규칙을 걸어 일부 경로의 write/edit 을 차단한다.

**무엇을 시연하나**
- **Pattern A — GuardedBackend** (FilesystemBackend 서브클래싱):
  - `/notes/ok.md` → 허용 → write 성공
  - `/etc/passwd` → deny → `WriteResult(error=...)` 반환, 에이전트가 거부 메시지 받음
- **Pattern B — PolicyWrapper** (제네릭 래퍼, 어떤 백엔드든 감쌀 수 있음):
  - StateBackend 위에 같은 deny 규칙 적용
  - `/notes/scratch.md` 허용 / `/private/secret.md` 거부

**관전 포인트**
- A-2 출력에 `Writes are not allowed under /etc/passwd` 류 메시지가 보여야 한다
- A-2 직후 호스트 디스크 확인에서 `/etc/passwd` 가 `exists=False` 여야 한다 (정책이 실제로 막았다는 증거)
- B-2 도 같은 형식의 거부 메시지가 나와야 한다 — Filesystem 이 아닌 State 위에도 같은 정책이 동작한다는 증명

**왜 그런가**
- `BackendProtocol` 의 `write`/`edit` 는 결과 자리에 `error=...` 를 채워 정책 위반을 표현하는 규약 — 예외를 던지지 않음
- **GuardedBackend** 는 한 백엔드(여기선 Filesystem) 에만 정책을 박는 가장 단순한 패턴
- **PolicyWrapper** 는 같은 정책을 State/Store/Composite 등 다른 백엔드에도 그대로 적용 가능 — *정책*과 *매체*가 분리됨
- 미들웨어 레이어(예: `PIIMiddleware`) 는 *내용 기반* 정책에 쓰고, 백엔드 레이어는 *경로 기반* 정책에 쓴다 (직교)

→ 자세한 패턴·운영 결정 휴리스틱은 교안 §8 (`archives/source/01_textbook.md`)
→ 소스: `archives/scripts_py/05_policy_hooks.py`

In [38]:
!python -u ../archives/scripts_py/05_policy_hooks.py

▶ Policy hooks 데모 — 백엔드 위에 권한(deny) 규칙을 걸어 일부 경로의 write/edit 을 차단.
  · Pattern A: GuardedBackend  — FilesystemBackend 서브클래싱 (가장 단순)
  · Pattern B: PolicyWrapper   — BackendProtocol 제네릭 래퍼 (어떤 백엔드든 감쌀 수 있음)
  · 두 패턴 모두 deny 매칭 시 WriteResult(error=...) 반환 → 에이전트가 에러를 메시지로 받음.

deny_prefixes = ['/etc/', '/private/']

sandbox root: /private/var/folders/rt/yfdr6vp909qbw4ybcq5bzffh0000gn/T/tmpjuyitqye

=== A-1) GuardedBackend: 허용 경로 /notes/ok.md write ===
  ↳ /notes/* 는 deny 목록에 없음 → 정상 write 되어야 함.

  → 호스트 디스크: /private/var/folders/rt/yfdr6vp909qbw4ybcq5bzffh0000gn/T/tmpjuyitqye/notes/ok.md exists=True  (True 여야 함)

=== A-2) GuardedBackend: 거부 경로 /etc/passwd write ===
  ↳ /etc/* 는 deny 매칭 → WriteResult(error=...) 반환 → 에이전트 답변에 'Writes are not allowed' 류 메시지가 보여야 함.
Writes are not allowed under /etc/passwd. I cannot modify system files.
  → 호스트 디스크: /private/var/folders/rt/yfdr6vp909qbw4ybcq5bzffh0000gn/T/tmpjuyitqye/etc/passwd exists=False  (False 여야 함 — 정책이 실제로 막았다는 증거)

=== B-1) PolicyWr